<a href="https://colab.research.google.com/github/Linqiaoqiao2/xAI_Project_DG_Test-Time-Augmentation/blob/no_domainbed_baseline/tta_pacs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Important First Step in Colab:**
Make sure to enable a GPU runtime for faster training:
1.  Go to **Runtime** in the Colab menu.
2.  Select **Change runtime type**.
3.  Choose **GPU** from the Hardware accelerator dropdown and click **Save**.



---

### Cell 1: Setup - Install Libraries (if needed) and Imports

In [ ]:
# PyTorch and Torchvision are usually pre-installed in Colab.
# If you need a specific version or other libraries, uncomment and run:

# !pip install pillow numpy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import numpy as np
import os
import zipfile
from PIL import Image
from tqdm.notebook import tqdm # For progress bars in Colab

# Check if GPU is available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# For reproducibility (optional, but good practice)
torch.manual_seed(42)
np.random.seed(42)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(42)
PACS_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
PACS_CATEGORIES = ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']

# ========================
# 🔧 Experiment Hyperparameters
# ========================
MODEL_CHOICE = 'resnet18'
PRETRAINED = True
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 3e-5

# 👇 This helps identify logs/models when saving
print(f"Using device: {DEVICE}, Model: {MODEL_CHOICE}, Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")


Using device: cuda
Using device: cuda, Model: resnet18, Epochs: 10, Batch: 32, LR: 3e-05


---

### Cell 2: Download and Prepare PACS Dataset

The PACS dataset needs to be downloaded. Here's a common way to get it. We'll download it directly into the Colab environment. If you have it on Google Drive, you can mount your Drive and adjust the path.

In [ ]:
# Cell 2: Load and Extract PACS Dataset (Ultra-Minimal - Command Style)

import os
import zipfile
from google.colab import drive

# --- Configuration (Must be perfectly correct and file must exist at the specified path) ---
GOOGLE_DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/PACS.zip"
COLAB_ZIP_NAME = "PACS.zip"
BASE_PACS_PATH = "kfold" # This must be the main folder name after unzipping
# --- End Configuration ---

# 1. Mount Google Drive
# Using force_remount to ensure a fresh mount attempt each time
drive.mount('/content/drive', force_remount=True)

# 2. Copy from Google Drive
os.system(f"cp '{GOOGLE_DRIVE_FILE_PATH}' {COLAB_ZIP_NAME}")

# 3. Unzip
with zipfile.ZipFile(COLAB_ZIP_NAME, 'r') as zip_ref:
    zip_ref.extractall(".") # Extract to current directory

# Keeping this final print so you know the value of BASE_PACS_PATH:
print(f"Dataset base path is configured as: '{BASE_PACS_PATH}'")

Mounted at /content/drive
Dataset base path is configured as: 'kfold'


---

### Cell 3: PACS_Dataset Class and Transformations





In [ ]:
from pathlib import Path
# Assuming Dataset, Image, transforms are imported in Cell 1 or globally available

class PACS_Dataset(Dataset):
    def __init__(self, base_path, domain_name, categories, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for label_idx, category in enumerate(categories):
            category_path = os.path.join(base_path, domain_name, category)
            if not os.path.isdir(category_path):
                continue
            for img_file in os.listdir(category_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(category_path, img_file))
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# --- Data Transformations  ---
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# With Augmentation
tta_transforms = [
    # Orininal image
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    # HorizontalFlip
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    #Rotation
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
]

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

---

### Cell 4: Model Definition (ResNet 18)

In [ ]:
def get_model(num_classes=len(PACS_CATEGORIES), pretrained=True):
    # Load the ResNet18 model with optional ImageNet pretrained weights
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)

    # Replace the final fully connected layer to match the number of target classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model
# Test model creation
try:
    test_model = get_model(num_classes=len(PACS_CATEGORIES))
    print("Model created successfully.")
    # print(test_model)  # Uncomment this line to view the model architecture
except Exception as e:
    print(f"Error creating model: {e}")

Model created successfully.


---

### Cell 5: Training and Evaluation Functions

In [ ]:
# Train
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in tqdm(dataloader, desc="Train", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

# Evaluation
def eval_one_epoch_tta(model, dataloader, criterion, device, tta_transforms, epoch_desc="Evaluating (TTA)"):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    from tqdm import tqdm
    progress_bar = tqdm(dataloader, desc=epoch_desc, leave=False)

    with torch.no_grad():
        for inputs, labels in progress_bar:
            labels = labels.to(device)
            batch_outputs = []

            # Key TTA
            for t in tta_transforms:
                augmented_inputs = torch.stack([t(img.cpu()) for img in inputs])  # Apply transform to each image
                augmented_inputs = augmented_inputs.to(device)
                outputs = model(augmented_inputs)
                batch_outputs.append(outputs)  #Average

            # 平均所有变换后的输出
            avg_logits = torch.stack(batch_outputs).mean(0)
            loss = criterion(avg_logits, labels)

            probs = torch.softmax(avg_logits, dim=1)
            preds = probs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)


            tqdm.write("", end="")  # force flush for notebooks



    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


---

### Cell 6: Leave-One-Domain-Out Evaluation Pipeline

In [ ]:
import json
import os
import torch
from torch.utils.data import DataLoader

# === Base path to PACS dataset ===
BASE_PACS_PATH = "kfold"

# === To store best validation accuracy for each target domain ===
all_accuracies = []

# === Leave-One-Domain-Out training and evaluation ===
for target_domain in PACS_DOMAINS:
    print(f"\n===== Target Domain = {target_domain} =====")

    # Define source domains: all domains except the target
    source_domains = [d for d in PACS_DOMAINS if d != target_domain]

    # Combine datasets from all source domains for training
    train_datasets = [
        PACS_Dataset(BASE_PACS_PATH, domain, PACS_CATEGORIES, transform=train_transform)
        for domain in source_domains
    ]
    train_dataset = torch.utils.data.ConcatDataset(train_datasets)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # Use target domain as validation set (not used in training)
    val_dataset = PACS_Dataset(BASE_PACS_PATH, target_domain, PACS_CATEGORIES, transform=None)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Create model, loss function, and optimizer
    model = get_model(len(PACS_CATEGORIES), PRETRAINED).to(DEVICE)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    best_val_acc = 0.0  # Track best validation accuracy on target domain

    # Start training
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = eval_one_epoch_tta(model, val_loader, criterion, DEVICE, tta_transforms)

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}")

        best_val_acc = max(best_val_acc, val_acc)

    #  Save best accuracy for this target domain
    all_accuracies.append(best_val_acc)

# 📈 Compute summary statistics
average_acc = sum(all_accuracies) / len(all_accuracies)
worst_acc = min(all_accuracies)

print("\n===== Cross-Domain Evaluation Summary =====")
for domain, acc in zip(PACS_DOMAINS, all_accuracies):
    print(f"{domain}: {acc:.3f}")
print(f"\nAverage Accuracy: {average_acc:.3f}")
print(f"Worst-case Accuracy: {worst_acc:.3f}")

# 💾 Save results to disk as JSON
os.makedirs("results", exist_ok=True)



===== Target Domain = art_painting =====


Train:   0%|          | 0/249 [00:00<?, ?it/s]

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'PIL.Image.Image'>

In [ ]:
import matplotlib.pyplot as plt

# 如果你在另一个 Cell 运行，请确保这两个变量已定义
# 示例：
# PACS_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
# all_accuracies = [0.812, 0.774, 0.855, 0.723]  # 用你自己训练得到的值替换

average_acc = sum(all_accuracies) / len(all_accuracies)
worst_acc = min(all_accuracies)

plt.style.use('ggplot')
plt.figure(figsize=(8, 5))
bars = plt.bar(PACS_DOMAINS, all_accuracies, color='skyblue')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.3f}', ha='center', va='bottom')

plt.axhline(average_acc, color='blue', linestyle='--', label='Average Accuracy')
plt.axhline(worst_acc, color='red', linestyle=':', label='Worst-case Accuracy')

plt.title("Accuracy per Domain")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()
